<a href="https://colab.research.google.com/github/leok4943-maker/11/blob/master/nb/Qwen3_(0_6B)-Phone_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

代码块 1：安装依赖

In [1]:
%%capture
!pip install -U unsloth
!pip install -U trl datasets peft accelerate bitsandbytes

In [6]:
%%capture
!pip install -U unsloth
!pip install -U "torchao==0.16.0"
!pip install -U trl datasets peft accelerate bitsandbytes transformers

In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU: Tesla T4


代码块 2：加载 Qwen3-0.6B 模型

In [8]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = torch.float16,   # Colab T4 推荐用 float16
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen3-0.6B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


代码块 3：添加 LoRA

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


代码块 4：创建训练数据

In [10]:
import json
from pathlib import Path

SYSTEM_PROMPT = """你是一个中文信息抽取器。
请从用户文本中抽取信息，并且只输出合法 JSON。
字段固定为：姓名、电话、时间、科室。
缺失字段用 null。
不要解释，不要输出 Markdown。"""

train_examples = [
    {
        "input": "张三，电话 13800000000，明天下午三点预约牙科。",
        "output": {"姓名": "张三", "电话": "13800000000", "时间": "明天下午三点", "科室": "牙科"},
    },
    {
        "input": "李女士想周五上午去皮肤科，手机号是 13911112222。",
        "output": {"姓名": "李女士", "电话": "13911112222", "时间": "周五上午", "科室": "皮肤科"},
    },
    {
        "input": "王小明预约内科，时间还没定，电话 13688889999。",
        "output": {"姓名": "王小明", "电话": "13688889999", "时间": None, "科室": "内科"},
    },
    {
        "input": "帮赵先生挂明天上午的骨科，电话没有留下。",
        "output": {"姓名": "赵先生", "电话": None, "时间": "明天上午", "科室": "骨科"},
    },
    {
        "input": "陈晨想预约眼科，时间是后天下午，手机号 13500001111。",
        "output": {"姓名": "陈晨", "电话": "13500001111", "时间": "后天下午", "科室": "眼科"},
    },
    {
        "input": "小刘电话 13722223333，周三上午看牙科。",
        "output": {"姓名": "小刘", "电话": "13722223333", "时间": "周三上午", "科室": "牙科"},
    },
    {
        "input": "孙先生要挂皮肤科，手机号没说，时间是下周一下午。",
        "output": {"姓名": "孙先生", "电话": None, "时间": "下周一下午", "科室": "皮肤科"},
    },
    {
        "input": "周五上午帮吴女士预约内科，电话是 13612344321。",
        "output": {"姓名": "吴女士", "电话": "13612344321", "时间": "周五上午", "科室": "内科"},
    },
    {
        "input": "帮我约一下明天下午的牙科，我叫陈晨，电话 13500001111。",
        "output": {"姓名": "陈晨", "电话": "13500001111", "时间": "明天下午", "科室": "牙科"},
    },
    {
        "input": "周三上午皮肤科，手机号 13922223333，姓名孙女士。",
        "output": {"姓名": "孙女士", "电话": "13922223333", "时间": "周三上午", "科室": "皮肤科"},
    },
    {
        "input": "小王想看骨科，电话没留，时间是后天。",
        "output": {"姓名": "小王", "电话": None, "时间": "后天", "科室": "骨科"},
    },
    {
        "input": "赵强手机号 13855556666，想下周二上午挂眼科。",
        "output": {"姓名": "赵强", "电话": "13855556666", "时间": "下周二上午", "科室": "眼科"},
    },
    {
        "input": "帮刘女士预约内科，时间还没确定，电话是 13700001111。",
        "output": {"姓名": "刘女士", "电话": "13700001111", "时间": None, "科室": "内科"},
    },
    {
        "input": "马先生想明天去看牙科，手机号 13677778888。",
        "output": {"姓名": "马先生", "电话": "13677778888", "时间": "明天", "科室": "牙科"},
    },
    {
        "input": "帮林小姐挂周六下午的皮肤科，没有留手机号。",
        "output": {"姓名": "林小姐", "电话": None, "时间": "周六下午", "科室": "皮肤科"},
    },
    {
        "input": "老周要预约骨科，电话 13900001234，时间是今天下午。",
        "output": {"姓名": "老周", "电话": "13900001234", "时间": "今天下午", "科室": "骨科"},
    },
    {
        "input": "预约眼科，姓名是何明，手机号暂时没有，时间下周五上午。",
        "output": {"姓名": "何明", "电话": None, "时间": "下周五上午", "科室": "眼科"},
    },
    {
        "input": "请帮小张预约明早内科，电话 13588889999。",
        "output": {"姓名": "小张", "电话": "13588889999", "时间": "明早", "科室": "内科"},
    },
    {
        "input": "王女士周日下午想看牙科，手机号 13899990000。",
        "output": {"姓名": "王女士", "电话": "13899990000", "时间": "周日下午", "科室": "牙科"},
    },
    {
        "input": "李雷想预约皮肤科，电话和时间都没有说。",
        "output": {"姓名": "李雷", "电话": None, "时间": None, "科室": "皮肤科"},
    },
]

eval_examples = [
    {
        "input": "刘洋想后天上午预约眼科，手机号 13712345678。",
        "output": {"姓名": "刘洋", "电话": "13712345678", "时间": "后天上午", "科室": "眼科"},
    },
    {
        "input": "帮马先生预约骨科，时间是明天下午，没有电话。",
        "output": {"姓名": "马先生", "电话": None, "时间": "明天下午", "科室": "骨科"},
    },
    {
        "input": "王女士想周六上午看皮肤科，电话 13988887777。",
        "output": {"姓名": "王女士", "电话": "13988887777", "时间": "周六上午", "科室": "皮肤科"},
    },
    {
        "input": "小陈要预约牙科，手机号还没给，时间是下周二下午。",
        "output": {"姓名": "小陈", "电话": None, "时间": "下周二下午", "科室": "牙科"},
    },
]

Path("/content/data").mkdir(exist_ok=True)

with open("/content/data/train.jsonl", "w", encoding="utf-8") as f:
    for item in train_examples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open("/content/data/eval.jsonl", "w", encoding="utf-8") as f:
    for item in eval_examples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("训练集数量:", len(train_examples))
print("验证集数量:", len(eval_examples))

训练集数量: 20
验证集数量: 4


代码块 5：加载数据集并格式化为 Chat 模板

In [11]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/data/train.jsonl",
        "eval": "/content/data/eval.jsonl",
    }
)

def format_example(example):
    user_text = example["input"]
    assistant_text = json.dumps(example["output"], ensure_ascii=False)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

    return {"text": text}

train_dataset = dataset["train"].map(format_example)
eval_dataset = dataset["eval"].map(format_example)

print(train_dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

<|im_start|>system
你是一个中文信息抽取器。
请从用户文本中抽取信息，并且只输出合法 JSON。
字段固定为：姓名、电话、时间、科室。
缺失字段用 null。
不要解释，不要输出 Markdown。<|im_end|>
<|im_start|>user
张三，电话 13800000000，明天下午三点预约牙科。<|im_end|>
<|im_start|>assistant
<think>

</think>

{"姓名": "张三", "电话": "13800000000", "时间": "明天下午三点", "科室": "牙科"}<|im_end|>



代码块 6：创建 Trainer

In [15]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",

    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,

        warmup_steps = 5,
        max_steps = 60,

        learning_rate = 2e-4,
        logging_steps = 1,

        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",

        seed = 3407,
        output_dir = "outputs",
        report_to = "none",

        fp16 = True,
        bf16 = False,

        # 关键修改：打开 packing，解决 padding_free + max_length 冲突
        packing = True,
        packing_strategy = "bfd",
        max_length = 512,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/20 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/20 [00:00<?, ? examples/s]

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Unsloth: Packing eval dataset (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


代码块 7：开始训练

In [16]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 30 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 5,046,272 of 601,096,192 (0.84% trained)


Step,Training Loss
1,3.874900
2,3.772800
3,3.738800
4,3.289000
5,2.940300
6,2.428300
7,2.068100
8,1.773800
9,1.500100
10,1.380300


TrainOutput(global_step=60, training_loss=0.6418735311056177, metrics={'train_runtime': 93.3351, 'train_samples_per_second': 2.571, 'train_steps_per_second': 0.643, 'total_flos': 190296733777920.0, 'train_loss': 0.6418735311056177})

代码块 8：训练后推理

In [17]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def extract_info(user_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        [prompt],
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        temperature=0.0,
    )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return result.strip()

In [18]:
test_text = "刘洋想后天上午预约眼科，手机号 13712345678。"
print(extract_info(test_text))

{"姓名": "刘洋", "电话": "13712345678", "时间": "后天上午", "科室": "眼科"}


代码块 9：批量测试

In [19]:
test_cases = [
    "刘洋想后天上午预约眼科，手机号 13712345678。",
    "帮马先生预约骨科，时间是明天下午，没有电话。",
    "王女士想周六上午看皮肤科，电话 13988887777。",
    "小陈要预约牙科，手机号还没给，时间是下周二下午。",
    "赵磊电话 13666668888，想明天上午预约内科。",
    "帮林小姐挂眼科，时间还没定，电话也没有。",
]

for text in test_cases:
    print("输入：", text)
    print("输出：", extract_info(text))
    print("-" * 80)

输入： 刘洋想后天上午预约眼科，手机号 13712345678。
输出： {"姓名": "刘洋", "电话": "13712345678", "时间": "后天上午", "科室": "眼科"}
--------------------------------------------------------------------------------
输入： 帮马先生预约骨科，时间是明天下午，没有电话。
输出： {"姓名": "马先生", "电话": null, "时间": "明天下午", "科室": "骨科"}
--------------------------------------------------------------------------------
输入： 王女士想周六上午看皮肤科，电话 13988887777。
输出： {"姓名": "王女士", "电话": "13988887777", "时间": "周六上午", "科室": "皮肤科"}
--------------------------------------------------------------------------------
输入： 小陈要预约牙科，手机号还没给，时间是下周二下午。
输出： {"姓名": "小陈", "电话": null, "时间": "下周二下午", "科室": "牙科"}
--------------------------------------------------------------------------------
输入： 赵磊电话 13666668888，想明天上午预约内科。
输出： {"姓名": "赵磊", "电话": "13666668888", "时间": "明天上午", "科室": "内科"}
--------------------------------------------------------------------------------
输入： 帮林小姐挂眼科，时间还没定，电话也没有。
输出： {"姓名": "林小姐", "电话": null, "时间": null, "科室": "眼科"}
--------------------------------------------------------

代码块 10：检查输出是否是合法 JSON

In [20]:
import json

REQUIRED_FIELDS = ["姓名", "电话", "时间", "科室"]

def check_json_output(text):
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return False, "不是合法 JSON"

    for field in REQUIRED_FIELDS:
        if field not in obj:
            return False, f"缺少字段：{field}"

    return True, obj


result = extract_info("王女士想周六上午看皮肤科，电话 13988887777。")
ok, parsed = check_json_output(result)

print("模型输出:", result)
print("是否合法:", ok)
print("解析结果:", parsed)

模型输出: {"姓名": "王女士", "电话": "13988887777", "时间": "周六上午", "科室": "皮肤科"}
是否合法: True
解析结果: {'姓名': '王女士', '电话': '13988887777', '时间': '周六上午', '科室': '皮肤科'}


代码块 11：保存 LoRA Adapter

In [21]:
model.save_pretrained("/content/qwen3-json-extractor-lora")
tokenizer.save_pretrained("/content/qwen3-json-extractor-lora")

print("已保存到 /content/qwen3-json-extractor-lora")

已保存到 /content/qwen3-json-extractor-lora
